In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/khulasasndh/game-of-thrones-books/004ssb.txt
/kaggle/input/datasets/khulasasndh/game-of-thrones-books/005ssb.txt
/kaggle/input/datasets/khulasasndh/game-of-thrones-books/001ssb.txt
/kaggle/input/datasets/khulasasndh/game-of-thrones-books/002ssb.txt
/kaggle/input/datasets/khulasasndh/game-of-thrones-books/003ssb.txt


In [31]:
import gensim

In [32]:
import os

In [33]:
import nltk

In [34]:
from nltk import sent_tokenize,word_tokenize
from gensim.utils import simple_preprocess

In [35]:
import glob

In [36]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [37]:
base_path = "/kaggle/input/datasets/khulasasndh/game-of-thrones-books/"
file_pattern = os.path.join(base_path, "*.txt")
file_paths = glob.glob(file_pattern)

In [38]:
def read_got_books(base_path):
    """Read all Game of Thrones books"""
    file_paths = [
        f"{base_path}001ssb.txt",
        f"{base_path}002ssb.txt", 
        f"{base_path}003ssb.txt",
        f"{base_path}004ssb.txt",
        f"{base_path}005ssb.txt"
    ]
    
    all_text = ""
    for file_path in file_paths:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                all_text += content + "\n\n"
            print(f"✓ Loaded {os.path.basename(file_path)}")
        except Exception as e:
            print(f"✗ Error loading {os.path.basename(file_path)}: {e}")
    
    return all_text

In [39]:
print("Reading Game of Thrones books...")
got_text = read_got_books(base_path)

Reading Game of Thrones books...
✓ Loaded 001ssb.txt
✓ Loaded 002ssb.txt
✓ Loaded 003ssb.txt
✗ Error loading 004ssb.txt: 'utf-8' codec can't decode byte 0x93 in position 151: invalid start byte
✗ Error loading 005ssb.txt: 'utf-8' codec can't decode byte 0x92 in position 349: invalid start byte


In [40]:
len(got_text)

5714235

In [41]:
import re
def tokenize_sentences(text):
    text = re.sub(r'[^a-zA-Z\s\.\?\!]', ' ', text)
    text = text.lower()

    sentences=sent_tokenize(text)
    tokenized=[]
    for sent in sentences:
        words=word_tokenize(sent)
        if len(words)>2:
            tokenized.append(words)
    return tokenized

In [42]:
print("\nTokenizing text...")
tokenized_sentences = tokenize_sentences(got_text)
print(f"Created {len(tokenized_sentences)} sentences")


Tokenizing text...
Created 88559 sentences


In [43]:
from gensim.models import Word2Vec

In [44]:
model=Word2Vec(
    window=10,
    workers=4,
    vector_size=100,
    min_count=5,   
)

In [48]:
model.build_vocab(tokenized_sentences)

In [49]:
model.train(tokenized_sentences,total_examples=model.corpus_count,epochs=model.epochs)

(4012168, 5750430)

In [50]:
model.wv.most_similar('daenerys')

[('stormborn', 0.8597745895385742),
 ('unburnt', 0.8229385018348694),
 ('elia', 0.7720212340354919),
 ('bethany', 0.768994152545929),
 ('arwyn', 0.7680860161781311),
 ('martell', 0.7581902146339417),
 ('thorns', 0.7562041282653809),
 ('sole', 0.7534818053245544),
 ('gerion', 0.7502257227897644),
 ('eighth', 0.7468201518058777)]

In [51]:
model.wv.doesnt_match(['cersei','jaime','bronn','tyrion'])

'cersei'

In [52]:
model.wv['king']

array([ 0.8299468 ,  1.7730862 ,  0.7404366 , -0.30071387,  0.413164  ,
       -3.6504154 , -0.5817769 ,  2.2505686 , -3.2951975 , -3.536219  ,
        1.2589501 , -0.7663896 ,  0.8448699 ,  0.6000406 ,  0.00450599,
       -0.36341006, -0.52054375, -2.248334  ,  1.3423011 ,  0.8685136 ,
       -1.2792715 , -1.8435076 ,  1.8139893 , -0.8555734 , -2.1612935 ,
        1.7687657 , -0.52504396,  0.71740437, -0.15537994,  3.0922499 ,
        0.5066227 ,  3.9601085 ,  1.0659418 , -1.6744682 ,  0.7632071 ,
        1.1036901 , -0.90120506, -0.69743556,  1.3628092 , -2.8915377 ,
        1.2663782 ,  0.456375  ,  1.1971625 , -1.2522148 , -0.74155456,
       -1.476907  ,  1.3957936 , -1.7389438 , -0.04429516,  1.090977  ,
       -2.2137384 , -2.612324  , -0.982018  , -0.5670957 , -0.2394827 ,
       -0.7183753 ,  3.3782296 ,  1.4366395 , -1.3769832 ,  2.3558455 ,
        1.3357104 ,  0.4538755 , -0.74217784,  0.97771806, -0.8010037 ,
        1.1506306 , -0.12953275, -3.6151981 ,  0.49413574,  0.21

In [53]:
model.wv['king'].shape

(100,)

In [54]:
model.wv.get_normed_vectors()

array([[-0.0148843 , -0.02659193,  0.01369374, ..., -0.14372933,
         0.07038154,  0.11177311],
       [-0.01953313,  0.02209939,  0.10592206, ..., -0.08515863,
        -0.19117409, -0.01129054],
       [-0.05903924, -0.08290467,  0.00793011, ..., -0.16942358,
         0.24440017,  0.09574285],
       ...,
       [ 0.0224096 ,  0.14012237,  0.00514342, ..., -0.02650245,
         0.08266345, -0.00813094],
       [-0.06745252,  0.14068945,  0.025964  , ..., -0.03724191,
         0.0097834 ,  0.00231179],
       [-0.02727517,  0.14166264,  0.10924957, ..., -0.05432308,
        -0.02472657,  0.01247952]], dtype=float32)

In [55]:
model.wv.get_normed_vectors().shape

(8845, 100)

In [57]:
y=model.wv.index_to_key

In [58]:
y

['.',
 'the',
 'and',
 'a',
 'to',
 'of',
 'he',
 'his',
 'was',
 'i',
 'you',
 's',
 'in',
 'it',
 'her',
 'had',
 'she',
 'that',
 'as',
 'with',
 'him',
 '?',
 'but',
 'not',
 'for',
 'they',
 'said',
 'at',
 'on',
 'my',
 'is',
 'lord',
 'have',
 'be',
 'them',
 'no',
 'from',
 'me',
 'were',
 'would',
 'all',
 'your',
 'when',
 'ser',
 'so',
 'one',
 'if',
 'will',
 'could',
 'there',
 'we',
 'their',
 'man',
 'are',
 'up',
 'king',
 'what',
 'this',
 'did',
 't',
 'out',
 'back',
 'do',
 'been',
 'by',
 'or',
 'jon',
 'men',
 'more',
 'down',
 'well',
 'than',
 'like',
 'page',
 'tyrion',
 'who',
 'only',
 'father',
 'hand',
 'now',
 'see',
 'off',
 'd',
 'even',
 'never',
 'before',
 'old',
 'know',
 'can',
 'into',
 'too',
 'an',
 'told',
 'eyes',
 'black',
 'made',
 'thought',
 'll',
 'then',
 'lady',
 'arya',
 'some',
 'long',
 'time',
 'how',
 'through',
 'here',
 'over',
 'face',
 'brother',
 'come',
 'head',
 'boy',
 'bran',
 'where',
 'sansa',
 'might',
 'still',
 'us',
 

In [59]:
from sklearn.decomposition import PCA

In [60]:
pca=PCA(n_components=3)

In [61]:
X=pca.fit_transform(model.wv.get_normed_vectors())

In [62]:
X[:5]

array([[-0.23064134, -0.379174  , -0.46564144],
       [ 0.07525627, -0.573137  , -0.11035906],
       [ 0.18166439, -0.37017962, -0.13903001],
       [-0.02056982, -0.26600397, -0.15431872],
       [-0.448777  , -0.35426086, -0.41437003]], dtype=float32)

In [64]:
import plotly.express as px

In [68]:
fig=px.scatter_3d(X[:100],x=0,y=1,z=2,color=y[:100])
fig.show()